In [ ]:
import os
import sys
from pathlib import Path

# Set these before importing any `reva` modules. Update the placeholder paths
# to match your machine or shared Jupyter environment.
os.environ["REVA_DATA_ROOT"] = "/path/to/your/data/root"
os.environ["REVA_HF_CACHE_ROOT"] = "/path/to/your/hf_cache/root"
os.environ["REVA_CHECKPOINT_ROOT"] = "/path/to/your/checkpoints/root"
os.environ["REVA_EVAL_RESULTS_ROOT"] = "/path/to/your/eval_results/root"
os.environ["REVA_REGION_DATA_ROOT"] = "/path/to/your/region_data/root"
os.environ["REVA_DECONTAMINATION_ROOT"] = "/path/to/your/decontamination/root"
os.environ["REVA_GROUNDING_DINO_ROOT"] = "/path/to/your/groundingdino/root"
os.environ["REVA_VQAV2_ROOT"] = "/path/to/your/vqav2/root"
os.environ["REVA_TEST_IMAGES_ROOT"] = "/path/to/your/test_images/root"

hf_cache_root = Path(os.environ["REVA_HF_CACHE_ROOT"]).expanduser()
os.environ["HF_HOME"] = str(hf_cache_root)
os.environ["HF_HUB_CACHE"] = str(hf_cache_root / "hub")
os.environ["HF_DATASETS_CACHE"] = str(hf_cache_root / "datasets")
os.environ["TRANSFORMERS_CACHE"] = str(hf_cache_root / "hub")

for var_name in (
    "REVA_DATA_ROOT",
    "REVA_HF_CACHE_ROOT",
    "REVA_CHECKPOINT_ROOT",
    "REVA_EVAL_RESULTS_ROOT",
    "REVA_REGION_DATA_ROOT",
    "REVA_DECONTAMINATION_ROOT",
    "REVA_GROUNDING_DINO_ROOT",
    "REVA_VQAV2_ROOT",
    "REVA_TEST_IMAGES_ROOT",
    "HF_HOME",
    "HF_HUB_CACHE",
    "HF_DATASETS_CACHE",
    "TRANSFORMERS_CACHE",
):
    print(f"{var_name} = {os.environ.get(var_name)}")


def find_reva_project_root(start: Path) -> Path:
    override = os.environ.get("REVA_PROJECT_DIR")
    if override:
        return Path(override).expanduser().resolve()

    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "reva" / "evaluation.py").is_file() and (candidate / "reva" / "config.py").is_file():
            return candidate

    raise FileNotFoundError(
        "Could not locate the ReVA project root from the current working directory. "
        "Set REVA_PROJECT_DIR to your cloned repo path."
    )


PROJECT_ROOT = find_reva_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())


# ReVA significance testing: paired t-test and McNemar test

Compare **ReVA (global only / Stage 1)** vs **ReVA (global + regional / Stage 3)** using
one eval per system and **question-level paired differences** (see `significance_paired_ttest.py`).

Requires prediction JSONLs under each eval output directory (same files produced by
`stage1_global_lora_evaluation.ipynb` and `stage3_concat_lora_evaluation.ipynb`).

This notebook prints the paired-test summary, shows a richer dataframe view, and can
save or display per-benchmark significance plots.

No GPU needed. Dependencies: `numpy`, `scipy`, `pandas` (display only), `matplotlib` (plots).

## 1. Paths.

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path(".").resolve()


DATA_ROOT = Path(os.environ.get("REVA_PROJECT_DIR", str(Path.cwd())))

BASELINE_DIR = DATA_ROOT / "eval_results/global576_stage1"
FINAL_DIR = DATA_ROOT / "eval_results/concat576_full_pipeline"

ALPHA = 0.01
EXPORT_CSV = DATA_ROOT / "eval_results/significance_pairs.csv"
PLOT_DIR = DATA_ROOT / "eval_results/significance_plots"
SHOW_PLOTS = False

for label, p in (("baseline", BASELINE_DIR), ("final", FINAL_DIR)):
    print(f"{label}: {p}  exists={p.is_dir()}")
print(f"plot_dir: {PLOT_DIR}")
print(f"show_plots: {SHOW_PLOTS}")

## 2. Run paired tests, show the summary table, and build a dataframe view.

In [ ]:
import importlib
import sys

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import reva.significance_paired_ttest as sig_pt
importlib.reload(sig_pt)

results, vqa_note = sig_pt.collect_paired_results(BASELINE_DIR, FINAL_DIR, alpha=ALPHA)
if vqa_note:
    print(vqa_note)

sig_pt.print_results(results, ALPHA)
results_df = sig_pt.results_to_dataframe(results)
results_df

## 3. Save or display per-benchmark significance plots.

In [ ]:
plot_paths = sig_pt.plot_all_significance(
    results,
    alpha=ALPHA,
    plot_dir=PLOT_DIR,
    show=SHOW_PLOTS,
)

if plot_paths:
    print("Wrote significance plots:")
    for plot_path in plot_paths:
        print(f"  {plot_path}")
else:
    print("No plot files were written.")

plot_paths

In [ ]:
from IPython.display import Image, display

for plot_path in plot_paths:
    display(Image(filename=str(plot_path)))